# 01 — EDA e auditoria

Este notebook descreve a base fornecida. A EDA pode olhar a base completa porque não é usada aqui para escolher hiperparâmetros ou reportar performance final. A seleção supervisionada fica isolada no notebook 02.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.data_preprocessing import load_data, validate_data, audit_semantic_inconsistencies

df = load_data(ROOT / "data/raw/telco_customer_churn.xlsx")
validate_data(df)
df.shape

(7043, 21)

In [2]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,3325-['N' 'S' 'A' 'U' 'G'],Female,0,Yes,No,61,Yes,No,Fiber optic,No internet service,...,No internet service,No,No,No,Month-to-month,Yes,Mailed check,82.76,5045.65,No
1,3329-['L' 'E' 'Q' 'M' 'H'],Male,0,Yes,No,8,No,No phone service,Fiber optic,No,...,No,Yes,No,No,Month-to-month,Yes,Electronic check,56.44,445.25,Yes
2,6955-['P' 'W' 'Q' 'D' 'X'],Male,0,Yes,No,41,Yes,No,DSL,No,...,No,No,Yes,No,Two year,No,Mailed check,97.57,3995.44,Yes
3,1409-['S' 'L' 'G' 'M' 'D'],Male,1,Yes,No,46,Yes,No,DSL,No,...,No,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),110.33,5087.22,No
4,9364-['M' 'J' 'O' 'G' 'Z'],Female,0,Yes,No,33,Yes,No,DSL,No,...,No internet service,No,Yes,No,Two year,Yes,Credit card (automatic),63.35,2101.75,No


In [3]:
df.dtypes

customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object

In [4]:
print("Duplicatas completas:", df.duplicated().sum())
print("customerID duplicado:", df["customerID"].duplicated().sum())
print("\nChurn:")
display(df["Churn"].value_counts())
display(df["Churn"].value_counts(normalize=True).rename("proportion"))

Duplicatas completas: 0
customerID duplicado: 0

Churn:


Churn
No     4567
Yes    2476
Name: count, dtype: int64

Churn
No     0.648445
Yes    0.351555
Name: proportion, dtype: float64

In [5]:
audit, summary = audit_semantic_inconsistencies(df)
display(audit)
summary

,rule,count,pct_rows
0,MultipleLines: PhoneService=Yes + No phone ser...,621,0.088173
1,StreamingTV: has internet + No internet service,586,0.083203
2,OnlineBackup: has internet + No internet service,584,0.082919
3,OnlineSecurity: has internet + No internet ser...,574,0.081499
4,DeviceProtection: has internet + No internet s...,557,0.079086
5,StreamingMovies: has internet + No internet se...,549,0.077950
6,TechSupport: has internet + No internet service,527,0.074826
7,OnlineBackup: InternetService=No + Yes/No,0,0.000000
8,MultipleLines: PhoneService=No + Yes/No,0,0.000000
9,OnlineSecurity: InternetService=No + Yes/No,0,0.000000


{'rows': 7043,
 'total_rule_violations': 3998,
 'affected_customers': 3023,
 'affected_pct': 0.42922050262672157}

### Nota semântica

`No` não é tratado como sinônimo de `No internet service` ou `No phone service`. O primeiro indica ausência do serviço adicional; os outros indicam que o serviço-base necessário não existe. Combinações incompatíveis são sinalizadas, mas não corrigidas automaticamente.

In [6]:
for col in ["Contract", "PaymentMethod", "InternetService"]:
    display(pd.crosstab(df[col], df["Churn"], normalize="index").round(4))

Churn,No,Yes
Contract,,
Month-to-month,0.5650,0.4350
One year,0.7663,0.2337
Two year,0.7765,0.2235


Churn,No,Yes
PaymentMethod,,
Bank transfer (automatic),0.6684,0.3316
Credit card (automatic),0.6757,0.3243
Electronic check,0.5741,0.4259
Mailed check,0.6773,0.3227


Churn,No,Yes
InternetService,,
DSL,0.6434,0.3566
Fiber optic,0.6526,0.3474
No,0.6507,0.3493


In [7]:
display(df.groupby("Churn")[["tenure", "MonthlyCharges", "TotalCharges"]].agg(["mean","median"]).round(2))

tenure        MonthlyCharges        TotalCharges         
        mean median           mean median         mean   median
Churn                                                          
No     36.41   36.0          70.33  70.67      2562.14  2113.28
Yes    35.60   35.0          69.46  69.53      2474.50  2020.58